# FaceRankNet — Master Colab Notebook
**Feature-Decomposed Facial Beauty Prediction**  
Dataset: SCUT-FBP5500 | Framework: PyTorch + DGL + MediaPipe

---
**Drive folder structure:**
```
MyDrive/Colab Notebooks/FaceRankNet/FaceRankNet/
  ├── config.py, preprocessing.py, dataset.py, model.py ...
  ├── run_colab.ipynb
  └── cache/   <- auto-created, persists across sessions
```
Run cells **in order**.

In [ ]:
import subprocess, sys, os
import torch

!pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html --force-reinstall -q

# MediaPipe & CV
!pip install -q mediapipe>=0.10 opencv-python-headless

# Scientific stack
!pip install -q pandas scikit-learn tqdm matplotlib scipy

# DGL dependency
!pip install -q torchdata

# Kaggle dataset downloader
!pip install -q kagglehub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
torchvision 0.25.0+cpu requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.


In [ ]:

import sys, types

if 'dgl.graphbolt' not in sys.modules:
    _gb = types.ModuleType('dgl.graphbolt')
    _gb.load_graphbolt = lambda: None
    sys.modules['dgl.graphbolt'] = _gb
    print('✓ DGL graphbolt pre-patched')

import dgl, torch
print(f'DGL     : {dgl.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

# Sanity check: confirm DGL can move a graph to GPU
if torch.cuda.is_available():
    _g = dgl.graph(([0], [1]))
    _g = _g.to('cuda')
    print('✓ DGL CUDA graph test passed')
else:
    print('⚠ No GPU detected — re-check Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# ============================================================
# Cell 3 — Mount Drive, copy project files, download dataset
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split

# ── Paths ────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/Colab Notebooks/FaceRankNet/FaceRankNet'
PROJECT_ROOT = '/content/FaceRankNet'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(CACHE_DIR,    exist_ok=True)

# ── Copy .py files Drive -> local VM (faster imports) ────────
copied = []
for fname in os.listdir(DRIVE_ROOT):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(DRIVE_ROOT, fname),
                    os.path.join(PROJECT_ROOT, fname))
        copied.append(fname)
sys.path.insert(0, PROJECT_ROOT)
print(f'✓ Copied {len(copied)} files: {copied}')

# ── Download dataset ─────────────────────────────────────────
print('\nDownloading SCUT-FBP5500 ...')
KAGGLE_PATH = kagglehub.dataset_download(
    'pranavchandane/scut-fbp5500-v2-facial-beauty-scores')
print('Downloaded to:', KAGGLE_PATH)

# ── Locate images + labels ───────────────────────────────────
IMAGE_DIR   = os.path.join(KAGGLE_PATH, 'Images', 'Images')
LABELS_FILE = os.path.join(KAGGLE_PATH, 'labels.txt')

if not os.path.isdir(IMAGE_DIR):
    for root, _, files in os.walk(KAGGLE_PATH):
        if any(f.lower().endswith(('.jpg','.png')) for f in files):
            IMAGE_DIR = root; break
if not os.path.isfile(LABELS_FILE):
    for root, _, files in os.walk(KAGGLE_PATH):
        if 'labels.txt' in files:
            LABELS_FILE = os.path.join(root,'labels.txt'); break

n_img = len([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.png'))])
print(f'Image dir: {IMAGE_DIR}  ({n_img} images)')

# ── Parse labels, split, save CSVs ───────────────────────────
df = pd.read_csv(LABELS_FILE, sep=' ', header=None, names=['Filename','Rating'])
df['_b'] = df['Rating'].round(0).astype(int).clip(1,5)
train_df, test_df = train_test_split(df.drop(columns=['_b']),
    test_size=0.20, random_state=42, stratify=df['_b'])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

TRAIN_CSV     = f'{DRIVE_ROOT}/train_labels.csv'
TEST_CSV      = f'{DRIVE_ROOT}/test_labels.csv'
CACHE_TRAIN   = f'{CACHE_DIR}/train_landmarks.pkl'
CACHE_TEST    = f'{CACHE_DIR}/test_landmarks.pkl'
AVG_FACE_PATH = f'{CACHE_DIR}/avg_face.npy'
PSEUDO_PATH   = f'{CACHE_DIR}/pseudo_labels.pkl'
CHECKPOINT    = f'{DRIVE_ROOT}/checkpoint_best.pt'

train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV,   index=False)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'✓ All paths ready. CHECKPOINT: {CHECKPOINT}')

In [ ]:
# ============================================================
# Cell 4 — Extract & cache landmarks  (~20-40 min, saved to Drive)
# ============================================================
import pickle
from preprocessing import preprocess_dataset

print('Processing TRAIN set ...')
train_coords = preprocess_dataset(
    image_dir=IMAGE_DIR, csv_path=TRAIN_CSV, cache_path=CACHE_TRAIN)

print('\nProcessing TEST set ...')
test_coords = preprocess_dataset(
    image_dir=IMAGE_DIR, csv_path=TEST_CSV, cache_path=CACHE_TEST)

print(f'✓ Train: {len(train_coords)} | Test: {len(test_coords)} faces cached')

In [ ]:
# ============================================================
# Cell 5 — Universal Average Face & pseudo-labels
# ============================================================
import pickle, pandas as pd
from pseudo_labels import (
    compute_universal_average_face, compute_all_pseudo_labels,
    save_avg_face, save_pseudo_labels)

train_filenames = pd.read_csv(TRAIN_CSV)['Filename'].tolist()
with open(CACHE_TRAIN, 'rb') as f:
    train_coords = pickle.load(f)

coord_list   = [train_coords[f] for f in train_filenames if f in train_coords]
avg_face     = compute_universal_average_face(coord_list)
save_avg_face(avg_face, AVG_FACE_PATH)

pseudo_labels = compute_all_pseudo_labels(train_coords, avg_face, train_filenames)
save_pseudo_labels(pseudo_labels, PSEUDO_PATH)

s = train_filenames[0]
print(f'Sample pseudo-labels for "{s}":')
for organ, sc in pseudo_labels.get(s, {}).items():
    print(f'  {organ:12s}: {sc:.3f}')

In [ ]:
# ============================================================
# Cell 6 — Dataset & DataLoader
# ============================================================
import pickle
from pseudo_labels import load_pseudo_labels
from dataset import FaceDataset, PairDataset, make_face_loader, make_pair_loader

with open(CACHE_TRAIN,'rb') as f: train_coords = pickle.load(f)
with open(CACHE_TEST, 'rb') as f: test_coords  = pickle.load(f)
pseudo_labels = load_pseudo_labels(PSEUDO_PATH)

train_face_ds = FaceDataset(TRAIN_CSV, train_coords, pseudo_labels)
test_face_ds  = FaceDataset(TEST_CSV,  test_coords)
pair_ds       = PairDataset(train_face_ds)
pair_loader   = make_pair_loader(pair_ds, shuffle=True)
val_loader    = make_face_loader(test_face_ds, shuffle=False)

print(f'Train faces: {len(train_face_ds)} | Pairs: {len(pair_ds)} | Test: {len(test_face_ds)}')
batch_a, batch_b, mask = next(iter(pair_loader))
print('Batch A ratings shape:', batch_a['ratings'].shape)

In [ ]:
# ============================================================
# Cell 7 — Model instantiation + dry-run
# ============================================================
import torch
from model import FaceRankNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = FaceRankNet().to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Move subgraphs to device — requires CUDA-enabled DGL
model.eval()
with torch.no_grad():
    sg  = {k: v.to(device) for k, v in batch_a['subgraphs'].items()}
    out = model(sg)
print('global_score shape:', out['global_score'].shape)
print('organ_weights     :', out['organ_weights'].cpu().numpy().round(3))

In [ ]:

PROJECT_ROOT = '/content/FaceRankNet'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ============================================================
# Cell 8 — Training  (checkpoint saved to Drive)
# ============================================================
from train import train as run_training

trained_model = run_training(
    train_csv=TRAIN_CSV,
    test_csv=TEST_CSV,
    landmark_cache_train=CACHE_TRAIN,
    landmark_cache_test=CACHE_TEST,
    pseudo_labels_path=PSEUDO_PATH,
    checkpoint_path=CHECKPOINT,
    resume=True,
)
print('\n✓ Training complete.')

In [ ]:
import torch
from model import FaceRankNet

# Memuat model dari checkpoint di Drive
if os.path.exists(CHECKPOINT):
    print(f'Loading checkpoint from: {CHECKPOINT}')
    # Inisialisasi model
    trained_model = FaceRankNet().to(device)

    # Load state dict
    checkpoint_data = torch.load(CHECKPOINT, map_location=device)

    # Menangani jika checkpoint berisi dictionary (misal: state_dict, optimizer, epoch)
    if isinstance(checkpoint_data, dict) and 'model_state_dict' in checkpoint_data:
        trained_model.load_state_dict(checkpoint_data['model_state_dict'])
    else:
        trained_model.load_state_dict(checkpoint_data)

    trained_model.eval()
    print('✓ Model successfully loaded and set to evaluation mode.')
else:
    print(f'⚠ Error: Checkpoint tidak ditemukan di {CHECKPOINT}. Pastikan path sudah benar.')

In [ ]:
# ============================================================
# Cell 9 — Evaluation: PCC, MAE, DPD
# ============================================================
from evaluate import run_full_evaluation, validate_local_scores

trained_model.eval()
metrics = run_full_evaluation(trained_model, val_loader, device)
print('=' * 40)
print(f'  PCC : {metrics["pcc"]:.4f}')
print(f'  MAE : {metrics["mae"]:.4f}')
print(f'  DPD : {metrics["dpd"]:.4f}')
print('=' * 40)
valid = validate_local_scores(trained_model, train_face_ds, device)
print('Local scores:', 'PASS' if valid else 'FAIL')

In [ ]:
# ============================================================
# Cell 10 — Visualisation: organ bar chart + weight pie chart
# ============================================================
import torch, matplotlib.pyplot as plt, matplotlib, numpy as np
from dataset import collate_faces
from torch.utils.data import DataLoader

matplotlib.rcParams['figure.dpi'] = 120
colors = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']

sample_batch = next(iter(DataLoader(
    test_face_ds, batch_size=4, shuffle=False, collate_fn=collate_faces)))

trained_model.eval()
with torch.no_grad():
    sg  = {k: v.to(device) for k, v in sample_batch['subgraphs'].items()}
    out = trained_model(sg)

organ_names = list(out['local_scores'].keys())
n = len(sample_batch['filenames'])
fig, axes = plt.subplots(1, n, figsize=(4*n, 4), sharey=True)
if n == 1: axes = [axes]
for i, ax in enumerate(axes):
    scores = [out['local_scores'][o][i].item() for o in organ_names]
    ax.bar(organ_names, scores, color=colors, edgecolor='white')
    ax.axhline(out['global_score'][i].item(), color='black', ls='--', lw=1.2,
               label=f'Global={out["global_score"][i].item():.2f}')
    ax.axhline(sample_batch['ratings'][i].item(), color='red', ls=':', lw=1.2,
               label=f'GT={sample_batch["ratings"][i].item():.2f}')
    ax.set_ylim(1, 5); ax.set_title(sample_batch['filenames'][i], fontsize=7)
    ax.set_xticklabels([o.replace('_','\n') for o in organ_names], fontsize=7)
    ax.legend(fontsize=6)
fig.suptitle('FaceRankNet - Organ Scores', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/organ_scores_bar.png', bbox_inches='tight')
plt.show()

with torch.no_grad():
    sw = torch.softmax(trained_model.fusion_weights, dim=0).cpu().numpy()
fig2, ax2 = plt.subplots(figsize=(6,6))
ax2.pie(sw, labels=[o.replace('_',' ').title() for o in organ_names],
        autopct='%1.1f%%', colors=colors, explode=[0.05]*len(organ_names), startangle=140)
ax2.set_title('Organ Importance Weights', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/organ_weights_pie.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figures saved to Drive.')